## 03 DUR 안전성 검증 (복약)

사용자가 복용 중인 약품들에 대해 식약처 공공 데이터 기반으로 약물 상호작용 및 안전성을 검증한다.

> 한국 DUR 시스템의 향후 발전 방향으로 환자 맞춤형 정보 제공이 제시된 바 있으며, 여기에는 만성질환자의 질환에 따른 DUR 정보 제공, 환자 특성별 용량 조절, 질병금기 정보 제공, 알레르기 및 약물 이상반응 이력 정보가 포함된다(HIRA, 2019). 본 노트북은 이 방향성의 일부 단면을 식약처 공공 데이터 기반으로 예시 수준에서 다루어 보고자 한다.

> **유의**: 본 노트북은 식약처 공공 데이터 기반의 교육·연구 목적 예제로 작성된 것이다. 검증 결과는 참고용 정보로 제공되며, 실제 복약 결정은 의·약사의 판단을 따르는 것이 권장된다.

검증 범위 (5겹):

| 순위 | 검증 유형 | HIRA 의원급 외래 점유율 | 데이터 소스 |
|---|---|---:|---|
| 1 | 동일성분 중복 | 42.2% | DRUG_INGREDIENT_MAP (블록 B에서 구축) |
| 2 | 효능군 중복 | 35.4% | `df_drug.분류명` |
| 3 | 노인주의 (정보 제공) | 14.7% | `df_dur_ingr` + 사용자 나이 |
| 4 | 1일 최대 투여량 초과 | (변경 건율 28.9%) | `df_dose` + DRUG_INGREDIENT_MAP |
| 5 | 회수약 (DUR 외 안전 체크) | — | `df_recall` (933건) |

1~3순위는 HIRA(2019) 표 4-31 의원급 외래 정보제공의 약 92%를 차지한다. 4·5순위는 빈도는 낮지만 임상 강도가 강해 보강한다.

사용 데이터 (01_eda에서 캐시):

| 키 | 별칭 | 행수 | 용도 |
|---|---|---:|---|
| 1_item_permit | df_drug | 43,276 | 약품 마스터 (효능군 = 분류명) |
| 2_item_permit_detail | df_detail | 43,276 | ATC 코드, 주성분명 |
| 5_easy_excel | df_easy | 4,762 | 일반의약품 친화 설명 (참고용) |
| 6_day_max_dosg | df_dose | 4,962 | 성분별 1일 최대 투여량 |
| 7_recall | df_recall | 933 | 회수·판매중지 |
| 10_dur_ingr | df_dur_ingr | 1,742 | DUR 성분 (노인주의 등) |
| 11_dur_item | df_dur_item | 약 404,000 | DUR 품목 (병용금기, 향후 PR — 식약처 갱신 주기에 따라 ±수백 행 변동) |

## 환경 설정 및 라이브러리 임포트

In [ ]:
"""03_dur_safety_check - DUR 안전성 검증.

사용자가 복용 중인 약품들에 대해 5겹 안전 검증을 수행한다.
동일성분 중복, 효능군 중복, 노인주의, 1일 최대 투여량 초과, 회수약을 검출하며,
HIRA(2019) 표 4-31의 의원급 외래 정보제공 우선순위(약 92%)를 따른다.
"""

# 표준 라이브러리
import os

# 서드파티 라이브러리
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

print(f'pandas: {pd.__version__}')
print(f'numpy: {np.__version__}')

## 환경 감지 및 경로 설정

02_drug_matching과 동일한 환경 감지. pickle 캐시 디렉토리(01_eda 산출물)도 함께 참조한다.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/AH_03_06_medication_data'
    PROCESSED_DIR = '/content/drive/MyDrive/AH_03_06_medication_processed'
    ENV = 'Colab'
except ImportError:
    BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '../../data/raw/medication'))
    PROCESSED_DIR = os.path.abspath(os.path.join(os.getcwd(), '../../data/processed/medication'))
    ENV = 'Local'

print(f'환경: {ENV}')
print(f'원본 데이터: {BASE_DIR}')
print(f'캐시 데이터: {PROCESSED_DIR}')

assert os.path.isdir(PROCESSED_DIR), (
    f'pickle 캐시 디렉토리가 없습니다: {PROCESSED_DIR}\n'
    f'먼저 01_eda 노트북을 실행해 캐시를 생성하세요.'
)
print('캐시 디렉토리 확인됨')

## 데이터 로드 (pickle 캐시)

03 작업에 필요한 7종 데이터프레임을 일괄 로드한다. 각 별칭은 함수 인자명·결과 키와 일관되도록 고정한다.

In [ ]:
# 03에 필요한 7종 데이터프레임을 pickle에서 일괄 로드
NEEDED_KEYS = [
    '1_item_permit',
    '2_item_permit_detail',
    '5_easy_excel',
    '6_day_max_dosg',
    '7_recall',
    '10_dur_ingr',
    '11_dur_item',
]

dfs = {}
for key in NEEDED_KEYS:
    pkl_path = os.path.join(PROCESSED_DIR, f'{key}.pkl')
    assert os.path.exists(pkl_path), f'pickle 파일 없음: {pkl_path}'

    df = pd.read_pickle(pkl_path)
    dfs[key] = df
    size_mb = os.path.getsize(pkl_path) / 1024**2
    print(f'  [{key}] {len(df):,}행 x {df.shape[1]}열 ({size_mb:.1f} MB)')

# 03 메인 데이터프레임 (별칭) - 함수 인자명과 일관성 유지
df_drug = dfs['1_item_permit']
df_detail = dfs['2_item_permit_detail']
df_easy = dfs['5_easy_excel']
df_dose = dfs['6_day_max_dosg']
df_recall = dfs['7_recall']
df_dur_ingr = dfs['10_dur_ingr']
df_dur_item = dfs['11_dur_item']

print(f'\n로드 완료. 총 {sum(len(df) for df in dfs.values()):,}행')

## 마스터 테이블 구축 (블록 B)

블록 C의 5겹 검증 함수가 빠르게 lookup할 수 있도록 식약처 데이터로부터 마스터 dict·set 6개를 미리 구축한다.

> 한국 의원급 의료기관의 2017년 외래 처방 DUR 분석에서 처방전간 동일성분 중복(42.2%), 효능군 중복(35.4%), 노인주의(14.7%)의 세 항목이 전체 정보제공의 약 92%를 차지하는 것으로 보고되어(HIRA, 2019), 본 노트북은 이 세 항목을 우선 다루어볼 영역으로 설정하였다.

본 블록에서 구축하는 마스터:

| 이름 | 타입 | 출처 | 용도 |
|---|---|---|---|
| `DRUG_TO_INGREDIENTS` | `dict[int, set[str]]` | df_detail.주성분명 (1차, 99.8% 커버) + df_dur_item.주성분 (2차 보완) | 1순위 동일성분 중복, 4순위 1일 최대량 |
| `INGREDIENT_CODE_TO_NAME` | `dict[str, str]` | 위 동일 (M코드 → 한글성분명) | 경고 메시지 한글명 표기 |
| `DRUG_TO_CLASS` | `dict[int, str]` | df_drug.분류명 | 2순위 효능군 중복 |
| `ELDERLY_CAUTION_CODES` | `set[str]` | HIRA §7-1 다빈도 노인주의 10종 → df_dose/df_detail 영문성분명 매칭으로 M코드 추출 | 3순위 노인주의 |
| `INGREDIENT_TO_MAX_DOSE` | `dict[str, dict]` | df_dose 성분코드별 집계 | 4순위 1일 최대량 초과 |
| `RECALLED_ITEM_SEQS` | `set[int]` | df_recall.품목기준코드 | 5순위 회수약 |

**노트 1 — DRUG_TO_INGREDIENTS 커버리지**: df_detail.주성분명은 식약처 마스터 43,276개 중 43,203개(99.8%)가 `[Mxxx]한글성분명` 형식을 보유한다. df_dur_item은 추가 보완용(DUR 전용 약품 일부).

**노트 2 — ELDERLY_CAUTION_CODES 데이터 출처 결정**: 식약처 raw 데이터(10_dur_ingr, 11_dur_item)는 DUR유형이 '병용금기' 전용이라 노인주의 직접 추출이 불가능하다. 본 블록은 HIRA(2019) 보고서 §7-1의 다빈도 노인주의 약물 상위 10종(amitriptyline·diazepam·clonazepam·nortriptyline·flunitrazepam·flurazepam·imipramine·chlordiazepoxide·clobazam·clomipramine)을 학술 근거로 채택하고, 6_day_max_dosg와 2_item_permit_detail의 영문성분명에서 매칭해 M코드를 추출한다. 향후 식약처 공식 노인주의 리스트가 갱신되면 별도 PR로 교체한다.

In [ ]:
import re

# BOM(\ufeff) 안전 컬럼 정규화 — 식약처 csv 일부 헤더에 BOM 존재 (예: '성분코드', '품목기준코드')
for _df in (df_dose, df_recall, df_dur_ingr):
    _df.columns = _df.columns.str.lstrip('\ufeff')

# 주성분명 format: [Mxxxxxx]한글성분명 (구분자 '/' 또는 '|')
#   df_detail.주성분명: '[M040702]포도당|[M040426]염화나트륨'
#   df_dur_item.주성분: '[M262752]소포스부비르/[M262910]레디파스비르'
INGREDIENT_PATTERN = re.compile(r'\[(M\d+)\]([^/\[|]+)')


def parse_ingredient_string(s) -> list:
    """[Mxxxxx]성분명 형식 문자열에서 (성분코드, 한글성분명) 페어 추출.

    단일제는 1개, 복합제는 '/' 또는 '|' 로 구분된 여러 개.

    Args:
        s: 주성분명 같은 형식의 문자열 (NaN/빈문자열 허용)

    Returns:
        [(성분코드, 한글성분명), ...] 페어 리스트. 빈 입력은 빈 리스트.
    """
    if pd.isna(s) or not s:
        return []
    return INGREDIENT_PATTERN.findall(str(s))


# 같은 품목일련번호가 여러 행에 등장할 수 있어 set으로 집계
DRUG_TO_INGREDIENTS: dict = {}      # int -> set[str]
INGREDIENT_CODE_TO_NAME: dict = {}  # str -> str

# 1차: df_detail.주성분명 (43,203개 약품 = 99.8% 커버)
print('1차 빌드: df_detail.주성분명 파싱 ...')
for _, row in (
    df_detail[['품목일련번호', '주성분명']]
    .dropna(subset=['주성분명'])
    .drop_duplicates()
    .iterrows()
):
    item_seq = int(row['품목일련번호'])
    pairs = parse_ingredient_string(row['주성분명'])
    if not pairs:
        continue
    if item_seq not in DRUG_TO_INGREDIENTS:
        DRUG_TO_INGREDIENTS[item_seq] = set()
    for code, name in pairs:
        DRUG_TO_INGREDIENTS[item_seq].add(code)
        if code not in INGREDIENT_CODE_TO_NAME:
            INGREDIENT_CODE_TO_NAME[code] = name.strip()

after_1st = len(DRUG_TO_INGREDIENTS)
print(f'  1차 후: DRUG_TO_INGREDIENTS = {after_1st:,}개 약품')

# 2차: df_dur_item.주성분 (DUR 전용 약품 보완)
print('2차 빌드: df_dur_item.주성분 파싱 ...')
for _, row in (
    df_dur_item[['품목일련번호', '주성분']]
    .dropna(subset=['주성분'])
    .drop_duplicates()
    .iterrows()
):
    item_seq = int(row['품목일련번호'])
    pairs = parse_ingredient_string(row['주성분'])
    if not pairs:
        continue
    if item_seq not in DRUG_TO_INGREDIENTS:
        DRUG_TO_INGREDIENTS[item_seq] = set()
    for code, name in pairs:
        DRUG_TO_INGREDIENTS[item_seq].add(code)
        if code not in INGREDIENT_CODE_TO_NAME:
            INGREDIENT_CODE_TO_NAME[code] = name.strip()

added = len(DRUG_TO_INGREDIENTS) - after_1st
print(f'  2차 추가: +{added:,}개 약품')

print(f'\n최종 DRUG_TO_INGREDIENTS: {len(DRUG_TO_INGREDIENTS):,}개 약품 ({len(DRUG_TO_INGREDIENTS)/len(df_drug)*100:.1f}% 커버)')
print(f'최종 INGREDIENT_CODE_TO_NAME: {len(INGREDIENT_CODE_TO_NAME):,}개 성분 한글명 등록됨')

# 샘플 검증
sample_drug = next(iter(DRUG_TO_INGREDIENTS))
sample_codes = DRUG_TO_INGREDIENTS[sample_drug]
sample_names = [INGREDIENT_CODE_TO_NAME.get(c, '?') for c in sample_codes]
print(f'\n샘플 매핑: 품목일련번호 {sample_drug} -> 성분코드 {list(sample_codes)} -> {sample_names}')

## 효능군 (분류명) + 노인주의 마스터

효능군 중복 검증(2순위)을 위해 약품별 분류명을 dict로 미리 추출한다. 분류명은 약효 카테고리 텍스트(예: "해열.진통.소염제")로 df_drug에 약품 1개당 1개씩 존재한다.

노인주의 검증(3순위)을 위해 노인주의 성분코드를 set으로 추출한다.

**데이터 출처 결정**: 식약처 raw 데이터(10_dur_ingr, 11_dur_item)는 DUR유형이 '병용금기' 전용이라 노인주의 직접 추출이 불가능하다. 따라서 HIRA(2019) 보고서 §7-1 표 4-29의 다빈도 노인주의 약물 상위 10종을 학술 근거로 채택하고, df_dose.성분명(영문) 직접 매칭(1차) + df_detail.영문성분명 매칭 후 주성분명의 [Mxxx] 패턴 추출(2차)로 M코드를 확보한다.

자료 §6-2에 따르면 노인주의는 변경 건율 1.8%로 차단보다 정보 제공 위주 등급(INFO)으로 설계할 학술적 근거가 있다(배성호 외 2021의 alert fatigue 문헌).

In [ ]:
# DRUG_TO_CLASS: 품목일련번호 -> 분류명 (예: "해열.진통.소염제")
DRUG_TO_CLASS: dict = (
    df_drug[['품목일련번호', '분류명']]
    .dropna(subset=['분류명'])
    .drop_duplicates(subset='품목일련번호')
    .set_index('품목일련번호')['분류명']
    .to_dict()
)
print(f'DRUG_TO_CLASS: {len(DRUG_TO_CLASS):,}개 약품 분류명 등록됨')

# ELDERLY_CAUTION_CODES: HIRA §7-1 다빈도 노인주의 약물 10종 → M코드
# raw 데이터(10_dur_ingr, 11_dur_item)에 노인주의 DUR 유형이 없어, 학술 자료 기반 매칭으로 대체
HIRA_ELDERLY_NAMES_EN = [
    'amitriptyline', 'diazepam', 'clonazepam', 'nortriptyline', 'flunitrazepam',
    'flurazepam', 'imipramine', 'chlordiazepoxide', 'clobazam', 'clomipramine',
]

ELDERLY_CAUTION_CODES: set = set()

# 1차: df_dose.성분명(영문) 매칭 → 성분코드 직접 추출
print('ELDERLY_CAUTION_CODES 1차: df_dose.성분명(영문) 매칭 ...')
dose_en = df_dose['성분명(영문)'].astype(str).str.lower()
for en in HIRA_ELDERLY_NAMES_EN:
    mask = dose_en.str.contains(en, na=False, regex=False)
    if mask.any():
        for code in df_dose.loc[mask, '성분코드'].astype(str):
            ELDERLY_CAUTION_CODES.add(code)

after_1st = len(ELDERLY_CAUTION_CODES)
print(f'  1차 후: {after_1st}개 M코드')

# 2차: df_detail.영문성분명 매칭 → 주성분명에서 [Mxxx] 추출 (df_dose 누락 보완)
print('ELDERLY_CAUTION_CODES 2차: df_detail 영문 매칭 → 주성분명 M코드 추출 ...')
detail_en = df_detail['영문성분명'].astype(str).str.lower()
for en in HIRA_ELDERLY_NAMES_EN:
    mask = detail_en.str.contains(en, na=False, regex=False)
    if mask.any():
        for ingr_str in df_detail.loc[mask, '주성분명'].dropna():
            for code, _ in INGREDIENT_PATTERN.findall(str(ingr_str)):
                ELDERLY_CAUTION_CODES.add(code)

added = len(ELDERLY_CAUTION_CODES) - after_1st
print(f'  2차 추가: +{added}개 M코드')
print(f'\n최종 ELDERLY_CAUTION_CODES: {len(ELDERLY_CAUTION_CODES)}개')

# Sanity check: HIRA §7-1 10종 각각의 매칭 상태
print('\n  HIRA §7-1 다빈도 노인주의 10종 매칭 상태:')
for en in HIRA_ELDERLY_NAMES_EN:
    found_dose = bool(dose_en.str.contains(en, na=False, regex=False).any())
    found_detail = bool(detail_en.str.contains(en, na=False, regex=False).any())
    flag = '[OK]' if (found_dose or found_detail) else '[X]'
    print(f'    {flag} {en}: dose={found_dose}, detail={found_detail}')

## 1일 최대 투여량 + 회수약 마스터

1일 최대량 검증(4순위)을 위해 성분코드별 1일최대투여량과 단위를 dict로 정리한다. 같은 성분이 여러 제형(정/캡슐/시럽 등)으로 등록된 경우 본 블록에서는 가장 큰 한계값(`max`)을 기준으로 한다. 향후 PR에서 제형별 분리 매핑으로 정밀화한다.

회수약 검증(5순위)을 위해 df_recall의 품목기준코드를 set으로 추출한다. 회수는 DUR 8유형에는 포함되지 않으나, 환자 안전에 직결되는 별도 안전 체크로 5순위에 둔다.

In [ ]:
# INGREDIENT_TO_MAX_DOSE: 성분코드 -> {max_dose, unit, name}
# 같은 성분 여러 제형이 있으면 max값을 한계로 (향후 PR에서 제형별 분리)
dose_grouped = df_dose.groupby('성분코드').agg(
    max_dose=('1일최대투여량', 'max'),
    unit=('투여단위', 'first'),
    name=('성분명(한글)', 'first'),
)
INGREDIENT_TO_MAX_DOSE: dict = {
    str(code): {
        'max_dose': float(row['max_dose']),
        'unit': str(row['unit']) if pd.notna(row['unit']) else '',
        'name': str(row['name']) if pd.notna(row['name']) else '',
    }
    for code, row in dose_grouped.iterrows()
}
print(f'INGREDIENT_TO_MAX_DOSE: {len(INGREDIENT_TO_MAX_DOSE):,}개 성분 1일 한계 등록')

# RECALLED_ITEM_SEQS: 회수약 품목일련번호 set
RECALLED_ITEM_SEQS: set = set(df_recall['품목기준코드'].astype(int))
print(f'RECALLED_ITEM_SEQS: {len(RECALLED_ITEM_SEQS):,}개 회수약')

# 마스터 6개 최종 요약
print('\n=== 블록 B 마스터 테이블 6개 요약 ===')
print(f'  1. DRUG_TO_INGREDIENTS:      {len(DRUG_TO_INGREDIENTS):>8,}개 약품 → 성분 집합')
print(f'  2. INGREDIENT_CODE_TO_NAME:  {len(INGREDIENT_CODE_TO_NAME):>8,}개 성분 한글명')
print(f'  3. DRUG_TO_CLASS:            {len(DRUG_TO_CLASS):>8,}개 약품 → 분류명')
print(f'  4. ELDERLY_CAUTION_CODES:    {len(ELDERLY_CAUTION_CODES):>8,}개 노인주의 성분')
print(f'  5. INGREDIENT_TO_MAX_DOSE:   {len(INGREDIENT_TO_MAX_DOSE):>8,}개 성분 1일 한계')
print(f'  6. RECALLED_ITEM_SEQS:       {len(RECALLED_ITEM_SEQS):>8,}개 회수약')

## 5겹 검증 함수 (블록 C)

블록 B의 마스터 6개를 활용해 5겹 안전 검증을 수행하는 함수 5개를 정의한다.

### 공통 시그니처

```python
def check_xxx(
    medications: list[dict],     # 입력: 약품 + 1일 복용량
    patient: dict | None = None, # 환자 컨텍스트 (age 등)
) -> list[dict]:                 # 출력: alert dict 리스트
```

`medications` 각 원소 형식: `{'item_seq': int, 'daily_amount': float, 'dose_unit': str}`

`patient` 형식: `{'age': int}` 또는 `None` — 본 블록에서는 age만 사용. 향후 PR에서 sex/pregnancy/weight/diseases/allergies 확장.

### Alert 등급 (배성호 외 2021 + HIRA 표 4-31 변경 건율)

| 등급 | 의미 | 적용 |
|---|---|---|
| `BLOCK` | 강한 차단 경고 | 동일성분 중복, 회수약 |
| `WARN` | 주의 경고 | 효능군 중복, 1일 최대량 초과 |
| `INFO` | 참고용 정보 | 노인주의 (변경 건율 1.8%, alert fatigue 회피) |

### Graceful skip 정책

데이터 미등록 케이스에서는 검증 근거가 부재한 상태이므로 본 노트북은 해당 항목에 대해 경고를 생성하지 않는다. 이는 근거 없는 경고로 인한 혼란을 피하기 위한 설계로 이해할 수 있으며, 다만 미등록 성분에 대한 위험을 놓칠 수 있다는 한계도 함께 가진다. 데이터 커버리지 확대가 향후 작업의 한 방향이 될 수 있다.

- `INGREDIENT_TO_MAX_DOSE` 미등록 성분 (48% 커버) → 용량 검증 skip
- 단위 불일치 (예: mg vs 정) → 용량 비교 skip, 향후 PR에서 단위 변환
- 환자 나이 미입력 → 노인주의 skip

In [ ]:
from collections import defaultdict

# === Alert 등급 (배성호 외 2021 + HIRA 표 4-31 변경 건율 기반) ===
LEVEL_BLOCK = 'BLOCK'  # 강한 차단 경고 (회수약, 동일성분 중복)
LEVEL_WARN = 'WARN'    # 주의 경고 (효능군 중복, 1일 최대량 초과)
LEVEL_INFO = 'INFO'    # 참고용 정보 (노인주의)

# === 노인 기준 연령 (HIRA 노인주의 정의) ===
ELDERLY_AGE_THRESHOLD = 65


# === 헬퍼 함수 ===

# 품목일련번호 -> 품목명 lookup용 dict (반복 lookup 성능 위해 미리 구축)
_DRUG_NAME_BY_SEQ: dict = (
    df_drug[['품목일련번호', '품목명']]
    .drop_duplicates(subset='품목일련번호')
    .set_index('품목일련번호')['품목명']
    .to_dict()
)


# 회수약 fallback용 lookup dict (df_drug에 없는 회수약 약품명 조회용)
_RECALL_NAME_BY_SEQ: dict = (
    df_recall[['품목기준코드', '품목명']]
    .drop_duplicates(subset='품목기준코드')
    .set_index('품목기준코드')['품목명']
    .to_dict()
)


def _get_drug_name(item_seq) -> str:
    """품목일련번호로 약품명 조회.

    Lookup 순서: df_drug 마스터 → df_recall 회수약 fallback → unknown(seq=...).
    일부 회수약은 제조중지·만료 등으로 df_drug에 부재하므로 df_recall에서 폴백.
    None 입력 시 '(미발견)' 반환 (시드 lookup 실패 안전망).
    """
    if item_seq is None:
        return '(미발견)'
    seq = int(item_seq)
    name = _DRUG_NAME_BY_SEQ.get(seq)
    if name is not None:
        return name
    return _RECALL_NAME_BY_SEQ.get(seq, f'unknown(seq={seq})')


def _format_ingredient_names(codes) -> str:
    """성분코드 iterable을 한글명 리스트로 포매팅 (정렬, 콤마 구분)."""
    names = sorted({INGREDIENT_CODE_TO_NAME.get(c, c) for c in codes})
    return ', '.join(names)


print(f'_DRUG_NAME_BY_SEQ: {len(_DRUG_NAME_BY_SEQ):,}개 약품명 lookup 준비됨')
print(f'_RECALL_NAME_BY_SEQ: {len(_RECALL_NAME_BY_SEQ):,}개 회수약명 fallback 준비됨')

## 1순위 동일성분 중복 + 2순위 효능군 중복

자료 §6 의원급 외래 정보제공의 1·2순위 (합산 77.6%).

**동일성분 중복** (BLOCK 등급)

두 약품이 같은 성분(식약처 의약품 성분 표준 코드인 M코드)을 가진 경우 검출한다. 한 약품에 성분이 여럿인 복합제는 두 약품의 성분 목록에 공통된 것이 하나라도 있으면 중복으로 처리한다. HIRA(2019)에서 이 유형의 변경 건율(경고 발생 후 의사가 실제 처방을 변경한 비율)이 15.9%로 가장 높아 강한 차단 등급을 적용한다.

**효능군 중복** (WARN 등급)

두 약품이 같은 효능 분류명(예: "해열.진통.소염제")에 속하는 경우 검출한다. 동일 성분이 아니어도 같은 효능 카테고리에 속하면 효과 중복이나 부작용 증폭 가능성이 있다. 변경 건율 9.3%로 동일성분 중복 다음으로 의미가 있으나, 즉각적인 차단보다는 주의 환기 성격이라 주의 등급으로 둔다.

In [ ]:
def check_concurrent_use(
    medications: list,
    patient: dict | None = None,
) -> list:
    """동일성분 중복 검증 (HIRA 1순위, 변경 건율 15.9%).

    여러 약품이 같은 성분코드(M코드)를 공유하면 동일성분 중복으로 BLOCK 등급 경고.
    복합제는 성분 집합 교집합으로 검출 (set intersection).

    Args:
        medications: [{'item_seq': int, 'daily_amount': float, 'dose_unit': str}, ...]
        patient: 본 함수 미사용 (시그니처 일관성)

    Returns:
        [{'level': 'BLOCK', 'type': 'concurrent_ingredient',
          'drugs': [seq_a, seq_b], 'ingredient_codes': [...],
          'ingredients': '한글명, ...', 'message': '...'}, ...]
    """
    alerts = []
    # 약품별 성분 집합 추출 (미등록 약품은 skip)
    drug_codes = {}
    for med in medications:
        seq = int(med['item_seq'])
        codes = DRUG_TO_INGREDIENTS.get(seq)
        if codes:
            drug_codes[seq] = codes

    # 약품 쌍별 교집합 확인 (O(N^2))
    seqs = list(drug_codes.keys())
    for i, seq_a in enumerate(seqs):
        for seq_b in seqs[i+1:]:
            overlap = drug_codes[seq_a] & drug_codes[seq_b]
            if overlap:
                names_overlap = _format_ingredient_names(overlap)
                alerts.append({
                    'level': LEVEL_BLOCK,
                    'type': 'concurrent_ingredient',
                    'drugs': [seq_a, seq_b],
                    'ingredient_codes': sorted(overlap),
                    'ingredients': names_overlap,
                    'message': (
                        f'동일성분 중복: {_get_drug_name(seq_a)}와 '
                        f'{_get_drug_name(seq_b)}가 공통 성분 [{names_overlap}]을 포함'
                    ),
                })
    return alerts


def check_class_duplication(
    medications: list,
    patient: dict | None = None,
) -> list:
    """효능군 중복 검증 (HIRA 2순위, 변경 건율 9.3% → WARN 등급).

    여러 약품이 같은 분류명(예: "해열.진통.소염제")을 공유하면 효능군 중복.
    """
    # 약품별 분류명 (미등록 약품 skip)
    drug_class = {}
    for med in medications:
        seq = int(med['item_seq'])
        cls = DRUG_TO_CLASS.get(seq)
        if cls:
            drug_class[seq] = cls

    # 분류명별 grouping
    by_class = defaultdict(list)
    for seq, cls in drug_class.items():
        by_class[cls].append(seq)

    alerts = []
    for cls, seqs in by_class.items():
        if len(seqs) >= 2:
            names = [_get_drug_name(s) for s in seqs]
            alerts.append({
                'level': LEVEL_WARN,
                'type': 'class_duplicate',
                'drugs': seqs,
                'class_name': cls,
                'message': f'효능군 중복 ({cls}): {", ".join(names)}',
            })
    return alerts


print('check_concurrent_use, check_class_duplication 정의 완료')

## 3순위 — 노인주의

자료 §6-2의 핵심 발견(노인주의 변경 건율 1.8%)을 반영한 INFO 등급 검증.

> DUR 시스템의 가장 큰 한계로 지적되는 점은 임상 맥락 없는 반복적 경고로 인한 alert fatigue이며(배성호 외, 2021), 이를 완화하기 위해 금기 등급별·진료과별 선별적 정보 제공이 권장된다(HIRA, 2019). 본 시스템은 이 권고를 반영하여 신뢰도 단계별 분기를 활용하고자 한다.

설계 원칙:

- 환자 나이 ≥ 65 인 경우에만 작동 (그 외는 graceful skip)
- 각 약품의 성분 ∩ `ELDERLY_CAUTION_CODES` (HIRA §7-1 10종 기반 13개 M코드) 존재 시 INFO 알림
- 차단(BLOCK)이 아닌 정보 제공(INFO) — 변경 건율 1.8% 학술 근거

In [ ]:
def check_elderly_warning(
    medications: list,
    patient: dict | None = None,
) -> list:
    """노인주의 검증 (HIRA 3순위, 변경 건율 1.8% → INFO 등급).

    환자 나이가 65세 이상일 때 ELDERLY_CAUTION_CODES (HIRA §7-1 10종)와 매칭.
    Alert fatigue 회피를 위해 차단(BLOCK)이 아닌 정보 제공(INFO) 등급 사용
    (배성호 외 2021, HIRA 2019 §6-2 권고 근거).

    Args:
        medications: [{'item_seq': int, 'daily_amount': float, 'dose_unit': str}, ...]
        patient: {'age': int} 형식. None이거나 age 없으면 검증 skip.

    Returns:
        [{'level': 'INFO', 'type': 'elderly_caution', 'drug': seq,
          'ingredient_codes': [...], 'ingredients': '...', 'message': '...'}, ...]
    """
    # 환자 컨텍스트 없으면 skip
    if patient is None or patient.get('age') is None:
        return []
    if patient['age'] < ELDERLY_AGE_THRESHOLD:
        return []

    alerts = []
    for med in medications:
        seq = int(med['item_seq'])
        codes = DRUG_TO_INGREDIENTS.get(seq, set())
        if not codes:
            continue
        hit = codes & ELDERLY_CAUTION_CODES
        if hit:
            names_hit = _format_ingredient_names(hit)
            alerts.append({
                'level': LEVEL_INFO,
                'type': 'elderly_caution',
                'drug': seq,
                'ingredient_codes': sorted(hit),
                'ingredients': names_hit,
                'message': (
                    f'노인주의 (참고): {_get_drug_name(seq)}에 노인주의 성분 '
                    f'[{names_hit}] 포함 (환자 {patient["age"]}세)'
                ),
            })
    return alerts


print('check_elderly_warning 정의 완료')

## 4순위 1일 최대 투여량 초과 + 5순위 회수약

**1일 최대량 초과** (WARN 등급)

약품의 각 성분에 대해 사용자의 1일 총 복용량(같은 성분이 여러 약품에 들어있으면 합산)을 `INGREDIENT_TO_MAX_DOSE`에 등록된 한계값과 비교한다. 변경 건율 28.9%로 임상 강도가 크다.

graceful skip 조건:

- `INGREDIENT_TO_MAX_DOSE`에 등록되지 않은 성분 (전체 성분의 약 52%) → 검증 skip
- 단위 불일치 (예: 약품은 mg 표기, 한계 데이터는 ml 표기) → 비교 skip
  → 향후 PR에서 단위 변환 매핑 추가가 검토될 수 있는 영역

**회수약** (BLOCK 등급)

DUR 8유형에는 포함되지 않으나 환자 안전에 직결되는 별도 안전 체크. 사용자 복용 약품의 품목일련번호가 회수 대상 목록(`RECALLED_ITEM_SEQS`)에 있는지 확인한다.

In [ ]:
def check_dose_limit(
    medications: list,
    patient: dict | None = None,
) -> list:
    """1일 최대 투여량 초과 검증 (HIRA 4순위, 변경 건율 28.9% → WARN 등급).

    각 약품의 성분별로 사용자 총 복용량(같은 성분 여러 약품 합산)을
    INGREDIENT_TO_MAX_DOSE의 max_dose와 비교한다.

    Graceful skip:
        - 1일 최대량 미등록 성분 (48% 커버 한계)
        - 단위 불일치 (mg vs 정 등 — 향후 PR에서 단위 변환)

    Args:
        medications: [{'item_seq': int, 'daily_amount': float, 'dose_unit': str}, ...]
        patient: 본 함수 미사용 (시그니처 일관성)
    """
    # 성분코드별 총 복용량 누적 (같은 단위 가정, 다르면 skip)
    ingredient_total: dict = defaultdict(lambda: {'total': 0.0, 'unit': '', 'drugs': []})

    for med in medications:
        seq = int(med['item_seq'])
        unit = med['dose_unit']
        amount = float(med['daily_amount'])
        codes = DRUG_TO_INGREDIENTS.get(seq, set())
        for code in codes:
            bucket = ingredient_total[code]
            # 단위가 처음 등록되거나 같으면 합산, 다르면 합산 skip 표시
            if not bucket['unit']:
                bucket['unit'] = unit
                bucket['total'] += amount
                bucket['drugs'].append(seq)
            elif bucket['unit'] == unit:
                bucket['total'] += amount
                bucket['drugs'].append(seq)
            else:
                # 단위 불일치 — graceful skip
                bucket['unit'] = '_MIXED'

    alerts = []
    for code, info in ingredient_total.items():
        if info['unit'] == '_MIXED':
            continue  # 단위 불일치 skip
        limit = INGREDIENT_TO_MAX_DOSE.get(code)
        if limit is None:
            continue  # 한계 미등록 skip
        if limit['unit'] != info['unit']:
            continue  # 한계와 사용자 단위 다름 skip
        if info['total'] > limit['max_dose']:
            name = INGREDIENT_CODE_TO_NAME.get(code, code)
            drug_names = [_get_drug_name(s) for s in info['drugs']]
            # 빈 단위·'-'·NaN은 메시지에서 생략 (df_dose 일부 데이터 한계)
            actual_unit = info['unit'] if info['unit'] not in ('', '-', 'nan') else ''
            limit_unit = limit['unit'] if limit['unit'] not in ('', '-', 'nan') else ''
            alerts.append({
                'level': LEVEL_WARN,
                'type': 'dose_exceeded',
                'drugs': info['drugs'],
                'ingredient_code': code,
                'ingredient_name': name,
                'actual': info['total'],
                'limit': limit['max_dose'],
                'unit': info['unit'],
                'message': (
                    f'{name} 1일 최대량 초과: '
                    f'총 {info["total"]:.1f}{actual_unit} > 한계 {limit["max_dose"]:.1f}{limit_unit} '
                    f'({", ".join(drug_names)})'
                ),
            })
    return alerts


def check_recall(
    medications: list,
    patient: dict | None = None,
) -> list:
    """회수약 검증 (DUR 외 안전 체크 → BLOCK 등급).

    품목일련번호가 RECALLED_ITEM_SEQS(823개) 에 있으면 회수 대상.
    """
    alerts = []
    for med in medications:
        seq = int(med['item_seq'])
        if seq in RECALLED_ITEM_SEQS:
            alerts.append({
                'level': LEVEL_BLOCK,
                'type': 'recall',
                'drug': seq,
                'message': f'회수 대상 약품: {_get_drug_name(seq)} (품목일련번호 {seq})',
            })
    return alerts


print('check_dose_limit, check_recall 정의 완료')

# 5겹 검증 함수 정의 확인
print('\n=== 블록 C 검증 함수 5개 정의 완료 ===')
for fn_name in [
    'check_concurrent_use',
    'check_class_duplication',
    'check_elderly_warning',
    'check_dose_limit',
    'check_recall',
]:
    print(f'  {fn_name}: {"[OK]" if fn_name in dir() else "[MISSING]"}')

## 통합 함수 (블록 D)

5겹 검증 함수를 일괄 호출하고 결과를 04 RAG 입력 컨텍스트로 활용 가능한 dict로 반환한다.

HIRA(2019) §5-1이 제시한 "환자 맞춤형 DUR 정보 제공" 방향에 따라 환자 컨텍스트(`patient`)를 인자로 받으며, `patient=None`인 경우 환자 의존 검증(노인주의)은 자동으로 skip된다.

### 출력 dict 구조

| 키 | 타입 | 내용 |
|---|---|---|
| `duplicates_ingredient` | list[dict] | 1순위 동일성분 중복 (BLOCK) |
| `duplicates_efficacy` | list[dict] | 2순위 효능군 중복 (WARN) |
| `elderly_cautions` | list[dict] | 3순위 노인주의 (INFO) |
| `dose_exceeded` | list[dict] | 4순위 1일 최대량 초과 (WARN) |
| `recall_warnings` | list[dict] | 5순위 회수약 (BLOCK) |
| `summary` | dict | 총 alert 수 + 등급별 count |

각 list 원소는 블록 C의 검증 함수 반환 형식 그대로 (level, type, drugs, message, ...).

In [ ]:
def safety_check_all(
    medications: list,
    patient: dict | None = None,
) -> dict:
    """5겹 DUR 안전 검증 통합 실행.

    HIRA(2019) §5-1이 제시한 환자 맞춤형 DUR 정보 제공 방향에 따라
    환자 컨텍스트(patient)를 인자로 받는다. patient=None인 경우
    환자 의존 검증(노인주의)은 자동 skip.

    Args:
        medications: [{'item_seq': int, 'daily_amount': float, 'dose_unit': str}, ...]
        patient: {'age': int} 또는 None

    Returns:
        04 RAG 입력 컨텍스트로 활용 가능한 dict.
        키: duplicates_ingredient, duplicates_efficacy, elderly_cautions,
            dose_exceeded, recall_warnings, summary
    """
    # 5겹 검증 일괄 호출
    dup_ingr = check_concurrent_use(medications, patient)
    dup_eff = check_class_duplication(medications, patient)
    elderly = check_elderly_warning(medications, patient)
    dose_exc = check_dose_limit(medications, patient)
    recall = check_recall(medications, patient)

    all_alerts = [*dup_ingr, *dup_eff, *elderly, *dose_exc, *recall]
    summary = {
        'total_alerts': len(all_alerts),
        'block_count': sum(1 for a in all_alerts if a['level'] == LEVEL_BLOCK),
        'warn_count': sum(1 for a in all_alerts if a['level'] == LEVEL_WARN),
        'info_count': sum(1 for a in all_alerts if a['level'] == LEVEL_INFO),
    }

    return {
        'duplicates_ingredient': dup_ingr,
        'duplicates_efficacy': dup_eff,
        'elderly_cautions': elderly,
        'dose_exceeded': dose_exc,
        'recall_warnings': recall,
        'summary': summary,
    }


# Smoke test: 빈 입력으로 호출해 dict 구조 정상 반환 확인
_smoke = safety_check_all([], None)
expected_keys = {
    'duplicates_ingredient', 'duplicates_efficacy', 'elderly_cautions',
    'dose_exceeded', 'recall_warnings', 'summary',
}
assert set(_smoke.keys()) == expected_keys, f'dict 키 불일치: {set(_smoke.keys())}'
assert _smoke['summary'] == {'total_alerts': 0, 'block_count': 0, 'warn_count': 0, 'info_count': 0}, (
    f'빈 입력 summary 비정상: {_smoke["summary"]}'
)
print('safety_check_all 정의 완료')
print(f'  빈 입력 결과 키: {sorted(_smoke.keys())}')
print(f'  빈 입력 summary: {_smoke["summary"]}')

## 시나리오 테스트 (블록 E)

5겹 검증 함수가 실제 식약처 데이터에서 학술 시드 기반 시나리오에 의도대로 작동하는지 검증한다. 자료 §7-1의 HIRA 다빈도 노인주의 약물 시드를 중심으로 5개 시나리오를 구성한다.

### 시나리오 5개

| # | 시나리오 | 학술 근거 | 검증 등급 |
|---|---|---|---|
| 1 | 노인주의 | HIRA §7-1 amitriptyline + 75세 | INFO |
| 2 | 동일성분 중복 | 자료 §6 1순위 (42.2%) | BLOCK |
| 3 | 효능군 중복 | 자료 §6 2순위 (35.4%) | WARN |
| 4 | 1일 최대량 초과 | 자료 §6 4순위 (변경률 28.9%) | WARN |
| 5 | 회수약 | DUR 외 안전 체크 | BLOCK |

### 시드 약품 lookup 정책

모든 시드는 마스터 6개(DRUG_TO_INGREDIENTS 등)에서 자동 lookup한다. 품목일련번호를 하드코딩하지 않으므로 식약처 데이터 버전 갱신에도 견고하다.

### 평가 기준

각 시나리오별로 기대 카테고리에 alert ≥ 1 발생하면 PASS. summary의 등급별 count가 5겹 설계와 일치하는지도 확인한다.

In [ ]:
# === 시드 약품 lookup 헬퍼 ===

def _lookup_drug_by_ingredient_en(en_name: str) -> int | None:
    """영문 성분명 키워드로 첫 매칭 약품(회수약 제외)의 품목일련번호 반환.

    시연 깔끔함을 위해 회수약은 우선순위에서 제외. 모두 회수약이면 None.
    """
    mask = df_detail['영문성분명'].astype(str).str.lower().str.contains(
        en_name.lower(), na=False, regex=False
    )
    if not mask.any():
        return None
    for seq in df_detail.loc[mask, '품목일련번호'].astype(int):
        if int(seq) not in RECALLED_ITEM_SEQS:
            return int(seq)
    return None  # 모두 회수약이면 None


def _find_second_drug_with_shared_ingredient(seq: int) -> int | None:
    """주어진 약품과 성분이 겹치는 다른 약품(회수약 제외)의 품목일련번호 반환."""
    codes = DRUG_TO_INGREDIENTS.get(seq, set())
    if not codes:
        return None
    for other_seq, other_codes in DRUG_TO_INGREDIENTS.items():
        if (other_seq != seq
                and other_seq not in RECALLED_ITEM_SEQS
                and codes & other_codes):
            return other_seq
    return None


def _find_two_drugs_same_class() -> tuple:
    """같은 분류명을 가진 두 약품(회수약 제외) 반환."""
    from collections import Counter
    active_class = {s: c for s, c in DRUG_TO_CLASS.items() if s not in RECALLED_ITEM_SEQS}
    class_counts = Counter(active_class.values())
    for cls, _cnt in class_counts.most_common():
        seqs = [s for s, c in active_class.items() if c == cls][:2]
        if len(seqs) >= 2:
            return seqs[0], seqs[1], cls
    return None, None, None


def _find_drug_with_dose_limit() -> tuple:
    """1일 최대량이 등록된 성분을 가진 약품(회수약 제외) + 한계값 반환.

    시연 깔끔함을 위해 단위가 비어있지 않은 성분 우선.
    """
    fallback = None
    for seq, codes in DRUG_TO_INGREDIENTS.items():
        if seq in RECALLED_ITEM_SEQS:
            continue
        for code in codes:
            if code in INGREDIENT_TO_MAX_DOSE:
                info = INGREDIENT_TO_MAX_DOSE[code]
                # 단위가 명시된 성분 우선 선호
                if info['unit'] and info['unit'] not in ('', '-', 'nan'):
                    return seq, info['unit'], info['max_dose'], info['name']
                if fallback is None:
                    fallback = (seq, info['unit'], info['max_dose'], info['name'])
    return fallback if fallback else (None, None, None, None)


def _find_two_active_drugs_from_hira() -> tuple:
    """HIRA §7-1 10종 중 비회수약 페어 ≥2 가능한 첫 성분 lookup.

    amitriptyline은 활성 약품 1개뿐이라 동일성분 중복 demo 불가.
    HIRA 10종을 우선순위 순으로 순회하며 활성 약품 ≥2개 있는 첫 성분 반환.

    Returns:
        (seq_a, seq_b, en_name) 또는 (None, None, None) — 페어 없으면.
    """
    for en in HIRA_ELDERLY_NAMES_EN:
        mask = df_detail['영문성분명'].astype(str).str.lower().str.contains(
            en, na=False, regex=False
        )
        active = []
        for seq in df_detail.loc[mask, '품목일련번호'].astype(int):
            seq_int = int(seq)
            if seq_int not in RECALLED_ITEM_SEQS:
                active.append(seq_int)
                if len(active) >= 2:
                    return active[0], active[1], en
    return None, None, None


# === 시드 구축 ===

print('=' * 70)
print('시드 lookup 중 ...')
print('=' * 70)

# 시나리오 1·2 시드: amitriptyline (HIRA §7-1 1순위)
amitri_seq = _lookup_drug_by_ingredient_en('amitriptyline')
print(f'  시나리오 1 amitriptyline: 품목일련번호 {amitri_seq} ({_get_drug_name(amitri_seq)})')

# 시나리오 2 시드: HIRA §7-1 10종 중 활성 페어 가능한 첫 성분
# amitriptyline은 활성 약품 1개뿐이라 동일성분 중복 demo 불가 (식약처 데이터 현황)
scen2_seq_a, scen2_seq_b, scen2_en = _find_two_active_drugs_from_hira()
print(f'  시나리오 2 동성분 페어 ({scen2_en}): {scen2_seq_a} ({_get_drug_name(scen2_seq_a)}), {scen2_seq_b} ({_get_drug_name(scen2_seq_b)})')

# 시나리오 3 시드: 같은 분류명 두 약품
cls_seq_a, cls_seq_b, cls_name = _find_two_drugs_same_class()
print(f"  시나리오 3 동일 분류명 '{cls_name}': {cls_seq_a}, {cls_seq_b}")

# 시나리오 4 시드: 1일 최대량 등록 성분 약품
dose_seq, dose_unit, dose_limit, dose_name = _find_drug_with_dose_limit()
print(f'  시나리오 4 한계 등록 약품: {dose_seq} ({dose_name}, 한계 {dose_limit}{dose_unit})')

# 시나리오 5 시드: 회수약 첫 원소
recall_seq = next(iter(RECALLED_ITEM_SEQS))
print(f'  시나리오 5 회수약: 품목일련번호 {recall_seq} ({_get_drug_name(recall_seq)})')


# === 시나리오 5개 정의 및 실행 ===

SCENARIOS = [
    {
        'name': '시나리오 1: 노인주의 (INFO)',
        'reference': 'HIRA §7-1 다빈도 노인주의 amitriptyline + 75세 환자',
        'expected_category': 'elderly_cautions',
        'medications': [
            {'item_seq': amitri_seq, 'daily_amount': 25.0, 'dose_unit': 'mg'},
        ],
        'patient': {'age': 75},
    },
    {
        'name': '시나리오 2: 동일성분 중복 (BLOCK)',
        'reference': f'자료 §6 1순위 (점유 42.2%) — HIRA §7-1 {scen2_en} 함유 활성 약품 2개',
        'expected_category': 'duplicates_ingredient',
        'medications': [
            {'item_seq': scen2_seq_a, 'daily_amount': 5.0, 'dose_unit': 'mg'},
            {'item_seq': scen2_seq_b, 'daily_amount': 5.0, 'dose_unit': 'mg'},
        ],
        'patient': None,
    },
    {
        'name': '시나리오 3: 효능군 중복 (WARN)',
        'reference': f"자료 §6 2순위 (점유 35.4%) — 분류명 '{cls_name}' 공유 2개 약",
        'expected_category': 'duplicates_efficacy',
        'medications': [
            {'item_seq': cls_seq_a, 'daily_amount': 1.0, 'dose_unit': '정'},
            {'item_seq': cls_seq_b, 'daily_amount': 1.0, 'dose_unit': '정'},
        ],
        'patient': None,
    },
    {
        'name': '시나리오 4: 1일 최대량 초과 (WARN)',
        'reference': f'자료 §6 4순위 (변경 건율 28.9%) — {dose_name} 한계 2배 복용',
        'expected_category': 'dose_exceeded',
        'medications': [
            {'item_seq': dose_seq, 'daily_amount': dose_limit * 2, 'dose_unit': dose_unit},
        ],
        'patient': None,
    },
    {
        'name': '시나리오 5: 회수약 (BLOCK)',
        'reference': 'DUR 외 안전 체크 — df_recall 첫 항목',
        'expected_category': 'recall_warnings',
        'medications': [
            {'item_seq': recall_seq, 'daily_amount': 1.0, 'dose_unit': '정'},
        ],
        'patient': None,
    },
]


print('\n' + '=' * 70)
print('시나리오 5개 실행')
print('=' * 70)

pass_count = 0
for sc in SCENARIOS:
    print(f"\n{sc['name']}")
    print(f"  학술 근거: {sc['reference']}")
    print(f"  입력: {len(sc['medications'])}개 약, patient={sc['patient']}")

    result = safety_check_all(sc['medications'], sc['patient'])
    target_alerts = result[sc['expected_category']]
    passed = len(target_alerts) >= 1

    status = '[PASS]' if passed else '[FAIL]'
    pass_count += 1 if passed else 0
    print(f"  결과: {status} ({sc['expected_category']} {len(target_alerts)}건)")
    print(f"  summary: {result['summary']}")

    # alert 메시지 출력
    for alert in target_alerts[:2]:  # 최대 2개만
        print(f"    [{alert['level']}] {alert['message']}")


# === 종합 평가 ===

print('\n' + '=' * 70)
print(f'시나리오 평가 종합: {pass_count}/{len(SCENARIOS)} PASS')
print('=' * 70)

## 작업 마무리 및 다음 단계

### 본 노트북에서 작성 결과 요

- 7종 식약처 pickle 캐시 일괄 로드 (총 502,905행)
- 마스터 dict/set 6개 구축 (`DRUG_TO_INGREDIENTS` 99.8% 커버 등)
- 5겹 검증 함수 5개 정의 (HIRA 표 4-31 우선순위 92.3% + 1일 최대량 + 회수약)
- 환자 컨텍스트(`patient`) 기반 노인주의 검증 (HIRA §5-1 "환자 맞춤형 DUR 정보 제공" 방향 구현)
- 통합 함수 `safety_check_all` — 04 RAG 입력 컨텍스트로 활용 가능한 dict 반환
- 시나리오 5개 자동 시드 기반 검증

### 평가 결과 (시나리오 5건)

| # | 시나리오 | 등급 | 상태 |
|---|---|---|---|
| 1 | 노인주의 (HIRA §7-1 amitriptyline) | INFO | PASS |
| 2 | 동일성분 중복 (자료 §6 1순위 42.2%) | BLOCK | PASS |
| 3 | 효능군 중복 (자료 §6 2순위 35.4%) | WARN | PASS |
| 4 | 1일 최대량 초과 (자료 §6 4순위 28.9%) | WARN | PASS |
| 5 | 회수약 (DUR 외 안전 체크) | BLOCK | PASS |


### 학술 인용 위치

| 인용                              | 셀 | 자료 §출처 |
|---------------------------------|---|---|
| HIRA 환자 맞춤형 DUR (정책 근거)         | cells[0] | §9 문장 1 (HIRA 2019 §5-1) |
| 의원급 외래 92.3% 우선순위               | cells[7] | §9 문장 2 (HIRA 2019 표 4-31) |
| Alert fatigue 회피 (노인주의 INFO 설계) | cells[17] | §9 문장 3 (배성호 외 2021) |

### 현재 한계 및 향후 PR

| 영역 | 한계 | 향후 작업 |
|---|---|---|
| 검증 범위 | DUR 8유형 중 5개만 구현 | 병용금기, 연령금기, 임부금기, 분할주의, 투여기간주의 추가 |
| 1일 최대량 | 성분 커버리지 48% (2,714/5,682) | 데이터 보강, 제형별 정밀 매핑 |
| 단위 변환 | mg vs 정 등 변환 미구현 | 단위 정규화 매핑 |
| 함수 시그니처 | dict 기반 | dataclass `PatientContext`·`SafetyResult` 리팩토링 |
| 시연 wrapper | 미구현 | 02 모듈화 후 `safety_check_by_name(drug_names, dosages, patient)` 추가 |

### 데이터 커버리지 요약

| 마스터 | 커버리지 | 비고 |
|---|---|---|
| DRUG_TO_INGREDIENTS | 43,203개 (99.8%) | df_detail 1차로 99.8% 커버, df_dur_item 2차는 실효 0건 (df_detail이 DUR 약품도 포함) |
| INGREDIENT_CODE_TO_NAME | 5,682개 | — |
| DRUG_TO_CLASS | 43,276개 (100%) | — |
| ELDERLY_CAUTION_CODES | 13개 M코드 | HIRA §7-1 10종 100% 매칭 |
| INGREDIENT_TO_MAX_DOSE | 2,714개 (48%) | 향후 데이터 보강 영역 |
| RECALLED_ITEM_SEQS | 823개 | — |

### 다음 노트북 (04 RAG/LLM)

- `safety_check_all` 반환 dict를 LLM prompt 입력 컨텍스트로 주입
- 임상진료지침 RAG로 환자 친화적 자연어 안내 생성
- ERD `CLINICAL_GUIDELINE` 마스터 + ChromaDB 청크 임베딩 활용
- HIRA §5-1 "약물 이상반응 이력 정보 / 알레르기 정보" 방향과 연계